# Multi-Tab Quantum Simulation GUI

This notebook provides a multi-tab graphical user interface (GUI) for quantum simulations using `ipywidgets`. The tabs allow you to configure the unit system, grid, potential, scattering, laser, time evolution, and current/charge calculations interactively.

## 1. Import Required Libraries

Import all necessary libraries, including numpy, matplotlib, ipywidgets, and custom modules such as `atomic_units` and `quantum_toolkit`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Custom modules (assumed to be available in your environment)
import atomic_units as au
import quantum_toolkit as quat
from quantum_toolkit import potentials as pots

ModuleNotFoundError: No module named 'atomic_units'

## 2. Create Tabbed GUI Layout

We use `ipywidgets.Tab` to create a multi-tab interface. Each tab will be populated with controls and visualizations for a specific aspect of the simulation.

In [ ]:
# Prepare tab children (to be filled in later)
tab_titles = [
    "Unit System", "Grid", "Potential", "Scatter", "Laser", "Time Evolution", "Current/Charge"
]
tab_children = [widgets.Output() for _ in tab_titles]

tabs = widgets.Tab(children=tab_children)
for i, title in enumerate(tab_titles):
    tabs.set_title(i, title)

display(tabs)

## 3. Unit System Tab: Set Atomic Units

This tab allows you to set and display atomic unit parameters (e.g., hbar, mass, charge). Calculated units and constants are shown for reference.

In [ ]:
with tab_children[0]:
    clear_output()
    # Widgets for atomic unit parameters
    hbar_widget = widgets.FloatText(value=1.0, description="ℏ (hbar):")
    mass_widget = widgets.FloatText(value=8.8, description="m₀ (mass):")
    kappa_widget = widgets.FloatText(value=2.3, description="κ₀ (kappa):")
    charge_widget = widgets.FloatText(value=3.4, description="e₀ (charge):")
    dx_widget = widgets.FloatText(value=0.05, description="dx (spacing):")
    calc_button = widgets.Button(description="Calculate Units")
    output_area = widgets.Output()

    def update_unit_system(_=None):
        with output_area:
            clear_output()
            unit_sys = au.AtomicUnitSystem(
                xh=hbar_widget.value,
                xe=charge_widget.value,
                xm=mass_widget.value,
                xk=kappa_widget.value
            )
            print("Fine structure constant:", unit_sys.fine_structure_constant)
            print("Energy unit [eV]:", unit_sys.energy_unit.to('eV'))
            print("Length unit [nm]:", unit_sys.length_unit.to('nm'))
            print("dx [nm]:", dx_widget.value * unit_sys.length_unit.to('nm'))

    calc_button.on_click(update_unit_system)

    display(widgets.VBox([
        widgets.HBox([hbar_widget, mass_widget, kappa_widget, charge_widget, dx_widget]),
        calc_button,
        output_area
    ]))
    update_unit_system()

## 4. Grid Tab: Configure Spatial Grid

Set grid parameters (number of points, spacing) and display the resulting grid and its properties.

In [ ]:
with tab_children[1]:
    clear_output()
    grid_points_widget = widgets.IntText(value=18*1024, description="Grid Points:")
    grid_dx_widget = widgets.FloatText(value=0.05, description="dx:")
    grid_update_button = widgets.Button(description="Update Grid")
    grid_output = widgets.Output()

    def update_grid(_=None):
        with grid_output:
            clear_output()
            uxgrid_num = grid_points_widget.value
            uxgrid_dx = grid_dx_widget.value
            uxgrid_width = uxgrid_num * uxgrid_dx
            uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)
            print(f"Grid type: {type(uxgrid)}")
            print(f"dx: {uxgrid_dx}")
            print(f"Grid width: {uxgrid_width}")
            plt.figure(figsize=(5,1))
            plt.plot(uxgrid, np.zeros_like(uxgrid), '.', markersize=1)
            plt.xlabel("x [arb. u.]")
            plt.title("Spatial Grid")
            plt.show()

    grid_update_button.on_click(update_grid)

    display(widgets.VBox([
        widgets.HBox([grid_points_widget, grid_dx_widget]),
        grid_update_button,
        grid_output
    ]))
    update_grid()

## 5. Potential Tab: Define Model Potential

Set potential parameters (heights, widths, types) and plot the resulting potential.

In [ ]:
with tab_children[2]:
    clear_output()
    wall_height_widget = widgets.FloatText(value=2.0, description="Wall Height [eV]:")
    wall_width_widget = widgets.FloatText(value=0.5, description="Wall Width [nm]:")
    well_depth_widget = widgets.FloatText(value=1.0, description="Well Depth [eV]:")
    well_width_widget = widgets.FloatText(value=10.0, description="Well Width [nm]:")
    potential_type_widget = widgets.Dropdown(
        options=["SmoothStackPotential", "ConstantPotential", "ZeroPotential"],
        value="SmoothStackPotential",
        description="Type:"
    )
    pot_update_button = widgets.Button(description="Plot Potential")
    pot_output = widgets.Output()

    def plot_potential(_=None):
        with pot_output:
            clear_output()
            # For demonstration, use a simple grid and unit system
            uxgrid = np.linspace(-20, 20, 1000)
            if potential_type_widget.value == "SmoothStackPotential":
                modelpot = pots.SmoothStackPotential(
                    uxgrid,
                    wallwidth=wall_width_widget.value,
                    wallheight=wall_height_widget.value,
                    welldepth=well_depth_widget.value,
                    wellwidth=well_width_widget.value,
                    wallrise=2,
                    wellfall=2
                )
                plt.plot(uxgrid, modelpot(0))
            elif potential_type_widget.value == "ConstantPotential":
                modelpot = pots.ConstantPotential(uxgrid, wall_height_widget.value)
                plt.plot(uxgrid, modelpot(0))
            else:
                modelpot = pots.ZeroPotential(uxgrid)
                plt.plot(uxgrid, modelpot(0))
            plt.xlabel("x [arb. u.]")
            plt.ylabel("Potential [arb. u.]")
            plt.title("Model Potential")
            plt.grid()
            plt.show()

    pot_update_button.on_click(plot_potential)

    display(widgets.VBox([
        widgets.HBox([
            wall_height_widget, wall_width_widget,
            well_depth_widget, well_width_widget,
            potential_type_widget
        ]),
        pot_update_button,
        pot_output
    ]))
    plot_potential()

## 6. Scatter Tab: Compute and Visualize Scattering

Set scattering parameters (energy range, k values), compute and plot transmission/reflection coefficients.

In [ ]:
with tab_children[3]:
    clear_output()
    energy_min_widget = widgets.FloatText(value=1.0, description="E min [eV]:")
    energy_max_widget = widgets.FloatText(value=6.0, description="E max [eV]:")
    num_points_widget = widgets.IntSlider(value=50, min=5, max=200, step=1, description="Points:")
    scatter_button = widgets.Button(description="Compute Scattering")
    scatter_output = widgets.Output()

    def compute_scattering(_=None):
        with scatter_output:
            clear_output()
            # Dummy calculation for demonstration
            e_vals = np.linspace(energy_min_widget.value, energy_max_widget.value, num_points_widget.value)
            t_vals = np.exp(-((e_vals - 3.5) ** 2) / 2)
            r_vals = 1 - t_vals
            plt.plot(e_vals, t_vals, "-o", label="Transmission")
            plt.plot(e_vals, r_vals, "-o", label="Reflection")
            plt.xlabel("Electron energy [eV]")
            plt.ylabel("Coefficient")
            plt.title("Scattering Coefficients")
            plt.legend()
            plt.grid()
            plt.show()

    scatter_button.on_click(compute_scattering)

    display(widgets.VBox([
        widgets.HBox([energy_min_widget, energy_max_widget, num_points_widget]),
        scatter_button,
        scatter_output
    ]))
    compute_scattering()

## 7. Laser Tab: Configure and Visualize Laser Potential

Set laser parameters (wavelength, amplitude, spot size) and plot the laser field in space and time.

In [ ]:
with tab_children[4]:
    clear_output()
    wavelength_widget = widgets.FloatText(value=800, description="Wavelength [nm]:")
    amplitude_widget = widgets.FloatText(value=1.0, description="Amplitude [GV/m]:")
    spot_size_widget = widgets.FloatText(value=800, description="Spot Size [nm]:")
    laser_plot_button = widgets.Button(description="Plot Laser Field")
    laser_output = widgets.Output()

    def plot_laser(_=None):
        with laser_output:
            clear_output()
            # Dummy laser field for demonstration
            x = np.linspace(-spot_size_widget.value/2, spot_size_widget.value/2, 500)
            t = np.linspace(0, 50, 200)
            X, T = np.meshgrid(x, t)
            omega = 2 * np.pi / wavelength_widget.value
            field = amplitude_widget.value * np.exp(-X**2/(2*(spot_size_widget.value/4)**2)) * np.sin(omega*T)
            plt.figure(figsize=(6,3))
            plt.pcolormesh(X, T, field, cmap='bwr')
            plt.colorbar(label="Field [GV/m]")
            plt.xlabel("x [nm]")
            plt.ylabel("time [arb. u.]")
            plt.title("Laser Field (space-time)")
            plt.show()

    laser_plot_button.on_click(plot_laser)

    display(widgets.VBox([
        widgets.HBox([wavelength_widget, amplitude_widget, spot_size_widget]),
        laser_plot_button,
        laser_output
    ]))
    plot_laser()

## 8. Time Evolution Tab: Run and Visualize Time Evolution

Run time evolution with selected parameters and visualize wavefunction evolution and probability density.

In [ ]:
with tab_children[5]:
    clear_output()
    time_steps_widget = widgets.IntSlider(value=100, min=10, max=1000, step=10, description="Time Steps:")
    run_te_button = widgets.Button(description="Run Time Evolution")
    te_output = widgets.Output()

    def run_time_evolution(_=None):
        with te_output:
            clear_output()
            # Dummy time evolution for demonstration
            x = np.linspace(-10, 10, 500)
            t = np.linspace(0, 10, time_steps_widget.value)
            X, T = np.meshgrid(x, t)
            psi = np.exp(-X**2/(2*2**2)) * np.cos(2*np.pi*T/10)  # Not a real solution!
            plt.figure(figsize=(6,3))
            plt.pcolormesh(X, T, psi, cmap='viridis')
            plt.colorbar(label="ψ(x,t)")
            plt.xlabel("x [arb. u.]")
            plt.ylabel("time [arb. u.]")
            plt.title("Wavefunction Evolution (demo)")
            plt.show()

    run_te_button.on_click(run_time_evolution)

    display(widgets.VBox([
        time_steps_widget,
        run_te_button,
        te_output
    ]))
    run_time_evolution()

## 9. Current/Charge Tab: Compute and Plot Probability Current and Charge

Select measurement points, compute and plot probability current and integrated charge over time.

In [ ]:
with tab_children[6]:
    clear_output()
    meas_left_widget = widgets.FloatText(value=-5.0, description="x left:")
    meas_right_widget = widgets.FloatText(value=5.0, description="x right:")
    current_button = widgets.Button(description="Plot Current/Charge")
    current_output = widgets.Output()

    def plot_current_charge(_=None):
        with current_output:
            clear_output()
            # Dummy current/charge for demonstration
            t = np.linspace(0, 10, 200)
            current_left = np.sin(0.5 * t)
            current_right = np.cos(0.5 * t)
            charge_left = np.cumsum(current_left) * (t[1]-t[0])
            charge_right = np.cumsum(current_right) * (t[1]-t[0])
            plt.figure(figsize=(6,3))
            plt.plot(t, current_left, label="Current left")
            plt.plot(t, current_right, label="Current right")
            plt.xlabel("time [arb. u.]")
            plt.ylabel("Current [arb. u.]")
            plt.title("Probability Current")
            plt.legend()
            plt.grid()
            plt.show()
            plt.figure(figsize=(6,3))
            plt.plot(t, charge_left, label="Charge left")
            plt.plot(t, charge_right, label="Charge right")
            plt.xlabel("time [arb. u.]")
            plt.ylabel("Charge [arb. u.]")
            plt.title("Integrated Charge")
            plt.legend()
            plt.grid()
            plt.show()

    current_button.on_click(plot_current_charge)

    display(widgets.VBox([
        widgets.HBox([meas_left_widget, meas_right_widget]),
        current_button,
        current_output
    ]))
    plot_current_charge()